# Token Streaming


Token Streaming with astream_events [Step 05 - Token-by-Token Output]

> **MLCourse - Agentic AI - LangGraph**

While `stream()` yields node-level updates, `astream_events` gives you
fine-grained access to individual tokens as the LLM generates them. This
notebook builds a simple agent and demonstrates how to capture token-by-token
output from the language model during graph execution.

# What you will learn

1. How `astream_events` differs from `stream()` in granularity.
2. The event types: `on_chat_model_stream` yields each token chunk.
3. Building a single-node agent graph that calls an LLM.
4. Collecting and printing tokens as they arrive.

### Key takeaways

- `stream()` = node-level updates (coarse).
- `astream_events` = event-level updates including individual tokens (fine).
- Token streaming enables real-time UI feedback and latency masking.

### Setup: imports, environment, LLM


In [ ]:
import os                           # env access
from dotenv import load_dotenv      # .env loading

load_dotenv(override=False)         # load keys without overriding

from typing import Annotated, TypedDict  # typed state

from langgraph.graph import StateGraph, END  # graph primitives

from langchain_ollama import ChatOllama  # local LLM
from langchain_core.messages import HumanMessage, SystemMessage  # message types


### Model guard: verify Ollama is running


In [ ]:
try:                                        # quick connectivity check
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("Ollama reachable -- token streaming will work")
except Exception as exc:
    LLM_AVAILABLE = False
    print("Ollama not reachable:", exc)
    print("LLM cells will produce simulated output instead")


### Define state: messages list with a system prompt slot


In [ ]:
class AgentState(TypedDict):
    """State holds the conversation messages for the agent."""
    messages: Annotated[list, "conversation messages"]


### Define a single agent node that calls the LLM


In [ ]:
def agent_node(state: AgentState) -> dict:
    """Agent node: builds messages and invokes the LLM."""
    # Construct the full message list with a system prompt
    msgs = [
        SystemMessage(content="You are a helpful assistant. Be concise."),
    ] + state["messages"]                   # append user messages from state
    llm = ChatOllama(model="llama3.1:8b", temperature=0)  # fresh LLM instance
    response = llm.invoke(msgs)             # invoke the LLM (non-streaming)
    return {"messages": [response.content]} # return assistant response


### Build a minimal graph: agent -> END


In [ ]:
builder = StateGraph(AgentState)            # graph with agent state
builder.add_node("agent", agent_node)      # single node: the LLM agent
builder.set_entry_point("agent")           # start at agent
builder.add_edge("agent", END)             # agent output goes to end

graph = builder.compile()                  # compile for execution

print("Graph compiled with 1 node: agent")


### Visualize the graph


In [ ]:
from IPython.display import Image, display

try:                                        # Mermaid rendering in try/except
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:                    # offline fallback
    print("Visualization unavailable:", exc)


### First: run with invoke() to show the baseline


In [ ]:
if LLM_AVAILABLE:
    result = graph.invoke({"messages": [HumanMessage(content="What is 2+2?")]})
    print("--- invoke() result ---")
    print(result["messages"][-1])           # print the final assistant message
else:
    print("SKIPPED: invoke() requires Ollama")


### Now: astream_events for token-level streaming


In [ ]:
if LLM_AVAILABLE:
    print("--- astream_events() token output ---")
    print()

    # Collect tokens as they arrive
    collected_tokens = []                   # accumulator for all token strings

    # astream_events returns an async generator of event dicts
    # We use version="v2" for the latest event schema
    import nest_asyncio                     # allow nested event loops in notebooks
    nest_asyncio.apply()

    async def stream_tokens():
        """Async helper to collect token events from the graph."""
        input_msgs = [HumanMessage(content="Explain recursion in one paragraph.")]

        async for event in graph.astream_events(
            {"messages": input_msgs},       # graph input
            version="v2",                   # event schema version
        ):
            kind = event["event"]           # event type string

            if kind == "on_chat_model_stream":
                chunk = event["data"]["chunk"]  # the AIMessageChunk
                token = chunk.content       # extract the text token
                if token:                   # skip empty tokens
                    collected_tokens.append(token)
                    print(token, end="", flush=True)  # print token inline

        print()                             # newline after streaming
        print()
        print("[collected %d tokens total]" % len(collected_tokens))

    import asyncio
    loop = asyncio.get_event_loop()
    loop.run_until_complete(stream_tokens())
else:
    print("SKIPPED: token streaming requires Ollama")


### Show how to filter events by name or tags


In [ ]:
if LLM_AVAILABLE:
    print("=== Filtered event streaming ===")
    print()

    # You can filter events by node name, event kind, or tags
    async def stream_filtered():
        input_msgs = [HumanMessage(content="Say hello in 3 words.")]

        async for event in graph.astream_events(
            {"messages": input_msgs},
            version="v2",
            include_types=["on_chat_model_stream"],
        ):
            chunk = event["data"]["chunk"]
            token = chunk.content
            if token:
                print(token, end="", flush=True)

        print()                             # final newline

    loop.run_until_complete(stream_filtered())

print()
print("NOTEBOOK COMPLETE: token streaming via astream_events demonstrated")
